In [8]:
import requests
import pandas as pd
import time
import os

# --- CONFIGURATION ---
EIA_API_KEY = "MtWUJ8RZsEzWmbrHZXgk7Nz5F7XzCztYuvVkIfYR"
BA_CODE = "TVA"
START_DATE = "2021-01-01T00"
END_DATE = "2025-12-31T23"
FILENAME = "tva_eia_21_25.csv"

def fetch_eia_master_data():
    all_data = []
    offset = 0
    rows_per_request = 5000
    
    print(f"Starting Ingestion for {BA_CODE}...")
    
    while True:
        url = f"https://api.eia.gov/v2/electricity/rto/region-data/data/?api_key={EIA_API_KEY}"
        params = {
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": BA_CODE,
            "facets[type][]": ["D", "DF", "NG", "TI"],
            "start": START_DATE,
            "end": END_DATE,
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "offset": offset,
            "length": rows_per_request
        }
        
        try:
            r = requests.get(url, params=params)
            res_json = r.json()
            if 'error' in res_json:
                print(f"EIA API Error: {res_json['error']}")
                break
            data = res_json['response']['data']
            if not data: break
            all_data.extend(data)
            print(f"Rows Ingested: {len(all_data)}...")
            if len(data) < rows_per_request: break
            offset += rows_per_request
            time.sleep(1) 
        except Exception as e:
            print(f"Error: {e}")
            break

    if not all_data: return None

    # Step 1: Create DataFrame
    df_raw = pd.DataFrame(all_data)
    
    # Step 2: Pivot
    df_pivot = df_raw.pivot_table(
        index='period', 
        columns='type-name', 
        values='value', 
        aggfunc='first'
    ).reset_index()
    
    # Step 3: Dynamic Mapping
    # This checks what names the EIA actually sent (sometimes they vary slightly)
    # and maps them to your specific required names.
    current_cols = df_pivot.columns.tolist()
    rename_map = {'period': 'timestamp'}
    
    # Matching logic to avoid KeyError
    for col in current_cols:
        if 'Demand Forecast' in col: rename_map[col] = 'demand_forecast_mwh'
        elif 'Net Generation' in col: rename_map[col] = 'net_generation_mwh'
        elif 'Net Interchange' in col: rename_map[col] = 'net_interchange_mwh'
        elif 'Demand' in col and 'Forecast' not in col: rename_map[col] = 'actual_demand_mwh'
        
    df_pivot = df_pivot.rename(columns=rename_map)
    
    # Step 4: Timezone Correction and Trimming
    df_pivot['timestamp'] = pd.to_datetime(df_pivot['timestamp'])
    df_pivot['timestamp'] = df_pivot['timestamp'] - pd.Timedelta(hours=5)
    
    mask = (df_pivot['timestamp'] >= '2021-01-01 00:00:00') & \
           (df_pivot['timestamp'] <= '2025-12-31 23:00:00')
    df_pivot = df_pivot.loc[mask].copy()
    
    # Step 5: Convert all columns after 'timestamp' to numeric
    for col in df_pivot.columns:
        if col != 'timestamp':
            df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')
            
    return df_pivot

# --- EXECUTE ---
df_eia = fetch_eia_master_data()

if df_eia is not None:
    save_path = os.path.join(os.getcwd(), FILENAME)
    df_eia.to_csv(save_path, index=False)
    print(f"\nSUCCESS: Data trimmed and saved to {save_path}")
    print("\nVerified Columns:")
    print(df_eia.columns.tolist())
    print(df_eia.head())

Starting Ingestion for TVA...
Rows Ingested: 5000...
Rows Ingested: 10000...
Rows Ingested: 15000...
Rows Ingested: 20000...
Rows Ingested: 25000...
Rows Ingested: 30000...
Rows Ingested: 35000...
Rows Ingested: 40000...
Rows Ingested: 45000...
Rows Ingested: 50000...
Rows Ingested: 55000...
Rows Ingested: 60000...
Rows Ingested: 65000...
Rows Ingested: 70000...
Rows Ingested: 75000...
Rows Ingested: 80000...
Rows Ingested: 85000...
Rows Ingested: 90000...
Rows Ingested: 95000...
Rows Ingested: 100000...
Rows Ingested: 105000...
Rows Ingested: 110000...
Rows Ingested: 115000...
Rows Ingested: 120000...
Rows Ingested: 125000...
Rows Ingested: 130000...
Rows Ingested: 135000...
Rows Ingested: 140000...
Rows Ingested: 145000...
Rows Ingested: 150000...
Rows Ingested: 155000...
Rows Ingested: 160000...
Rows Ingested: 165000...
Rows Ingested: 170000...
Rows Ingested: 175000...
Rows Ingested: 175130...

SUCCESS: Data trimmed and saved to /Users/garrett/Documents/CECS CAP 26/tva_eia_21_25.csv

In [9]:
actual_rows = len(df_eia)
expected_rows = 43824

if actual_rows == expected_rows:
    print(f"Perfect: You have all {actual_rows} hours.")
elif actual_rows < expected_rows:
    print(f"Missing Data: You are short {expected_rows - actual_rows} hours. Check for API timeouts.")
else:
    print(f"Duplicates Found: You have {actual_rows - expected_rows} extra rows. Check your timezone shift logic.")

Missing Data: You are short 28 hours. Check for API timeouts.


In [10]:
# Create a perfect reference range
perfect_range = pd.date_range(start='2021-01-01 00:00:00', end='2025-12-31 23:00:00', freq='h')

# Find what is in the perfect range but NOT in your dataframe
missing_timestamps = perfect_range.difference(df_eia['timestamp'])

print(f"Total Missing Hours: {len(missing_timestamps)}")
print("\nFirst 10 missing timestamps:")
print(missing_timestamps)

Total Missing Hours: 28

First 10 missing timestamps:
DatetimeIndex(['2025-12-30 20:00:00', '2025-12-30 21:00:00',
               '2025-12-30 22:00:00', '2025-12-30 23:00:00',
               '2025-12-31 00:00:00', '2025-12-31 01:00:00',
               '2025-12-31 02:00:00', '2025-12-31 03:00:00',
               '2025-12-31 04:00:00', '2025-12-31 05:00:00',
               '2025-12-31 06:00:00', '2025-12-31 07:00:00',
               '2025-12-31 08:00:00', '2025-12-31 09:00:00',
               '2025-12-31 10:00:00', '2025-12-31 11:00:00',
               '2025-12-31 12:00:00', '2025-12-31 13:00:00',
               '2025-12-31 14:00:00', '2025-12-31 15:00:00',
               '2025-12-31 16:00:00', '2025-12-31 17:00:00',
               '2025-12-31 18:00:00', '2025-12-31 19:00:00',
               '2025-12-31 20:00:00', '2025-12-31 21:00:00',
               '2025-12-31 22:00:00', '2025-12-31 23:00:00'],
              dtype='datetime64[ns]', freq='h')


In [11]:
df = pd.read_csv("tva_eia_21_25.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43796 entries, 0 to 43795
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp                  43796 non-null  object 
 1   Day-ahead demand forecast  43750 non-null  float64
 2   actual_demand_mwh          43772 non-null  float64
 3   Net generation             43772 non-null  float64
 4   Total interchange          43768 non-null  float64
dtypes: float64(4), object(1)
memory usage: 1.7+ MB
